In [ ]:
# 1. Install necessary libraries
!pip install flickrapi folium pandas

import flickrapi
import pandas as pd
import folium
import time
from google.colab import files
from IPython.display import display

# 2. Enter your Flickr API credentials (replace with your actual Key and Secret)
API_KEY = ''
API_SECRET = ''

# Initialize Flickr API
flickr = flickrapi.FlickrAPI(API_KEY, API_SECRET, format='parsed-json')

# 3. Set search parameters for Castlefield
# Precise bounding box: min_lon (west), min_lat (south), max_lon (east), max_lat (north)
castlefield_bbox = '-2.2625,53.4695,-2.2490,53.4785'

# Your 20 specific Castlefield keywords
castlefield_keywords = [
    'castlefield warehouse', 'castlefield industrial warehouse',
    'castlefield canal basin', 'castlefield canal',
    'castlefield railway viaduct', 'castlefield viaduct',
    'castlefield railway arch', 'castlefield goods yard',
    'castlefield freight depot', 'castlefield industrial heritage',
    'castlefield industrial building', 'castlefield warehouse manchester',
    'manchester warehouse castlefield', 'bridgewater canal castlefield',
    'rochdale canal castlefield', 'castlefield basin manchester',
    'castlefield rail infrastructure', 'castlefield canal warehouse',
    'castlefield logistics heritage', 'castlefield transport hub'
]
tags_string = ','.join(castlefield_keywords)

print(f"Searching Flickr for the following keywords: {tags_string}")
print(f"Using precise bounding box: {castlefield_bbox}")
print("Starting auto-pagination fetch for Castlefield, this may take some time...")

try:
    # 4. Use a loop for auto-pagination
    all_photos_dict = {}
    current_page = 1
    max_pages = 20

    while current_page <= max_pages:
        print(f"Fetching data for page {current_page}...")

        photos = flickr.photos.search(
            tags=tags_string,
            tag_mode='any',
            bbox=castlefield_bbox,
            has_geo=1,
            extras='geo,url_s',
            per_page=250,
            page=current_page
        )

        photo_list = photos['photos']['photo']

        if not photo_list:
            print("No more photos, fetching completed.")
            break

        for photo in photo_list:
            all_photos_dict[photo['id']] = photo

        total_pages = int(photos['photos']['pages'])
        if current_page >= total_pages:
            print(f"Reached the last page ({total_pages}), fetching completed.")
            break

        current_page += 1
        time.sleep(1)

    final_photo_list = list(all_photos_dict.values())
    print(f"\nSuccess! Fetched {len(final_photo_list)} unique photos related to Castlefield.")

    # 5. Convert data to Pandas DataFrame and clean
    if len(final_photo_list) > 0:
        df = pd.DataFrame(final_photo_list)
        df['latitude'] = df['latitude'].astype(float)
        df['longitude'] = df['longitude'].astype(float)

        df = df[(df['latitude'] != 0.0) & (df['longitude'] != 0.0)]

        # 6. Create pure dark interactive map with no labels
        print("Generating pure dark map with no labels...")

        # Set map center precisely to the middle of the provided bounding box
        castlefield_map = folium.Map(
            location=[53.4740, -2.2558],
            zoom_start=16,
            tiles='https://{s}.basemaps.cartocdn.com/dark_nolabels/{z}/{x}/{y}{r}.png',
            attr='&copy; OpenStreetMap contributors &copy; CARTO'
        )

        # Add each photo as a semi-transparent blue solid dot
        for idx, row in df.iterrows():
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=4,
                color='blue',
                weight=1,
                fill=True,
                fill_color='blue',
                fill_opacity=0.7
            ).add_to(castlefield_map)

        # 7. Export HTML and CSV files and auto-download
        print("Preparing to download files to your computer...")

        html_filename = 'castlefield_industrial_heritage_map.html'
        castlefield_map.save(html_filename)

        csv_filename = 'castlefield_industrial_coordinates.csv'
        export_df = df[['latitude', 'longitude']]
        export_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

        files.download(html_filename)
        files.download(csv_filename)

        print("All done! Please check your browser's download pop-ups.")
        print("Map preview below:")

        display(castlefield_map)

    else:
        print("No photos found matching these keywords in Castlefield.")

except flickrapi.exceptions.FlickrError as e:
    print(f"API call error: {e}")
    print("Please ensure your API_KEY and API_SECRET are correct.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

ERROR:flickrapi.auth.OAuthFlickrInterface:do_request: Status code 403 received, content:
ERROR:flickrapi.auth.OAuthFlickrInterface:    <html><body><h1>403 Forbidden</h1>
Request forbidden by administrative rules.
</body></html>



Searching Flickr for the following keywords: castlefield warehouse,castlefield industrial warehouse,castlefield canal basin,castlefield canal,castlefield railway viaduct,castlefield viaduct,castlefield railway arch,castlefield goods yard,castlefield freight depot,castlefield industrial heritage,castlefield industrial building,castlefield warehouse manchester,manchester warehouse castlefield,bridgewater canal castlefield,rochdale canal castlefield,castlefield basin manchester,castlefield rail infrastructure,castlefield canal warehouse,castlefield logistics heritage,castlefield transport hub
Using precise bounding box: -2.2625,53.4695,-2.2490,53.4785
Starting auto-pagination fetch for Castlefield, this may take some time...
Fetching data for page 1...
API call error: do_request: Status code 403 received
Please ensure your API_KEY and API_SECRET are correct.
